In [92]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re

# ── Set your city here ────────────────────────────────────────────────────────
CITY = "Haridwar-India"
# ─────────────────────────────────────────────────────────────────────────────

BASE_URL = "https://www.numbeo.com/cost-of-living/in/{city}"
OUTPUT_FILE = "numbeo_cost_of_living.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[–—]", "-", text)
    return text


def scrape_summary(soup: BeautifulSoup) -> dict:
    """Scrape family of four and single person monthly cost from summary box."""
    summary = {}

    box = soup.find("div", class_="inside_box_summary_content")
    if not box:
        print("  ⚠️  Summary box not found")
        return summary

    for li in box.find_all("li"):
        text = li.get_text(" ", strip=True)
        span = li.find("span", class_="emp_number")
        if not span:
            continue

        value_raw = span.get_text(strip=True).replace(",", "").replace("₹", "").replace("$", "").replace("€", "").replace("£", "").strip()
        try:
            value = float(value_raw)
        except ValueError:
            continue

        if "family of four" in text.lower():
            summary["monthly_cost_family_of_four"] = value
        elif "single person" in text.lower():
            summary["monthly_cost_single_person"] = value

    return summary


def scrape_city(city: str) -> dict:
    url = BASE_URL.format(city=city.replace(" ", "-"))
    print(f"Fetching: {url}")

    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    table = soup.find("table", class_="data_wide_table")
    if table is None:
        print(f"No data table found for {city}")
        return {}

    # Start row with city and summary costs first
    row = {"city": city}
    row.update(scrape_summary(soup))

    print(f"  → Family of four : ₹{row.get('monthly_cost_family_of_four', 'N/A')}")
    print(f"  → Single person  : ₹{row.get('monthly_cost_single_person', 'N/A')}")

    # Scrape all item prices from table
    for tr in table.find_all("tr"):
        price_td = tr.find("td", class_="priceValue")
        if price_td is None:
            continue

        tds = tr.find_all("td")
        if not tds:
            continue

        item_name = normalize(tds[0].get_text(strip=True))
        price_raw = (
            price_td.get_text(strip=True)
            .replace(",", "")
            .replace("₹", "")
            .replace("$", "")
            .replace("€", "")
            .replace("£", "")
            .strip()
        )

        try:
            row[item_name] = float(price_raw)
        except ValueError:
            row[item_name] = ""

    print(f"✓ {len(row) - 3} items scraped for {city}")  # -3 for city + 2 summary cols
    return row


def save_to_csv(row: dict, filepath: str):
    new_df = pd.DataFrame([row])

    if os.path.isfile(filepath) and os.path.getsize(filepath) > 0:
        existing_df = pd.read_csv(filepath)

        existing_df.columns = [
            normalize(c) if c not in ("city", "monthly_cost_family_of_four", "monthly_cost_single_person") else c
            for c in existing_df.columns
        ]

        all_cols = list(existing_df.columns) + [
            c for c in new_df.columns if c not in existing_df.columns
        ]
        existing_df = existing_df.reindex(columns=all_cols)
        new_df = new_df.reindex(columns=all_cols)

        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df

    combined_df.to_csv(filepath, index=False, encoding="utf-8-sig")
    print(f"💾 Saved → {filepath}  (total rows: {len(combined_df)})")


if __name__ == "__main__":
    data = scrape_city(CITY)
    if data:
        save_to_csv(data, OUTPUT_FILE)

Fetching: https://www.numbeo.com/cost-of-living/in/Haridwar-India
  ⚠️  Summary box not found
  → Family of four : ₹N/A
  → Single person  : ₹N/A
✓ 52 items scraped for Haridwar-India
💾 Saved → numbeo_cost_of_living.csv  (total rows: 150)
